# Creating a Simple Agent with Tracing

In [1]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [7]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [11]:
fitness_agent = Agent(
    name="Fitness Assistant",
    instructions="""
    You are a helpful assistant giving out fitness advice.
    You give concise answers, you give only free body weight exercises, and you give only exercises that can be done at home.
    """,
)

In [12]:
nutrition_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful assistant giving out nutrition advice.
    You give concise answers.
    """,
)

Let's execute the Agent:

In [10]:
with trace("fitness coach"):
    result = await Runner.run(fitness_agent, "What are some good home workouts for elderly people?")

print(result)

RunResult:
- Last agent: Agent(name="Fitness Assistant", ...)
- Final output (str):
    Here’s a simple, at-home routine using only body weight, suitable for many seniors. Start slow and use a chair or wall for support as needed.
    
    Warm-up (3–5 min)
    - March in place or walk around the room
    - Easy arm circles and shoulder rolls
    - Ankle circles while standing or sitting
    
    Strength and balance (2–3 sets each, 8–12 reps; 2–3 days/week)
    1) Sit-to-stand (chair-assisted squats)
    - From a chair, stand up fully, then sit down slowly. Use arms for support if needed.
    2) Wall push-ups
    - Stand at arm’s length from a wall, hands on wall shoulder-width apart, bend elbows to near chest, press back out.
    3) Seated leg extensions
    - Sit tall, extend one leg straight, hold, lower. Alternate legs.
    4) Standing hip abduction
    - Hold onto chair, lift leg to the side without tipping torso. Return slowly. Switch legs.
    5) Calf raises
    - Hold chair for

In [15]:
with trace("Simple Nutrition Agent dummy run"):
    result = await Runner.run(nutrition_agent, "How healthy are guavas?")

print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    Guavas are very healthy. Key points:
    
    - Very high in vitamin C (superior to many fruits) and also provide vitamin A, potassium, and fiber.
    - About 5–6 g fiber per 100 g; helps digestion and post-meal blood sugar control.
    - Antioxidants (carotenoids, flavonoids) support heart health and immunity.
    - Low to moderate calories (approx. 60–70 kcal per 100 g, depending on variety).
    
    Tips:
    - Eat the flesh and seeds for maximum fiber.
    - Pair with protein or fat to help absorption of fat-soluble vitamins.
    - Choose fresh, whole guava over juice with added sugar.
    
    Cautions:
    - Kidney disease or very high potassium diets may need moderation.
    - Added sugars in juice or processed products can negate benefits.
    
    Overall: a nutrient-dense fruit with notable immune and digestive benefits.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)


Streaming the answer to the screen, token by token

In [16]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas, guava, apple, litchi, mango? give me facts in depth")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Here are concise, evidence-based points for each fruit. Numbers are approximate per 100 g unless noted, and per-standard serving sizes are mentioned.

- Banana
  - Nutrients: ~89 kcal, 23 g carbs (bananas are higher in carbs), ~358 mg potassium, ~1.1 g protein, ~2.6 g fiber, vitamin B6, vitamin C.
  - Health highlights: good quick energy from natural sugars; decent potassium for blood pressure; provides vitamin B6 and some fiber.
  - Gut/antioxidants: contains resistant starch when unripe; contains polyphenols.
  - Considerations: moderate-to-high glycemic index; portion control if watching sugar/calories.
  - Typical serving: 1 medium banana (about 118 g).

- Guava
  - Nutrients: ~68 kcal, very high vitamin C (~228 mg/100 g), ~5.4 g fiber, modest potassium (~200 mg), folate.
  - Health highlights: exceptional vitamin C content; high fiber supports digestion and fullness; antioxidants like lycopene and flavonoids present.
  - Considerations: very sweet in some varieties but fiber helps

_Good Job!_